As programs grow, repeating the same logic in multiple places becomes a maintenance problem — fix a bug in one copy and forget the other three. Functions solve this by giving a name to a reusable block of code. You define the logic once and call it from anywhere. Functions also let you break a large problem into smaller, named pieces that are easier to read, test, and reason about. This notebook covers everything from defining a basic function to default arguments, variadic parameters, scope, lambdas, and treating functions as first-class values.

## Defining and Calling Functions

You define a function with the `def` keyword, a name, parentheses, a colon, and an indented body.

```python
def function_name(parameters):
    # body
    return value
```

- The function is **defined** once and **called** as many times as you like.
- A function that reaches the end without a `return` statement implicitly returns `None`.
- Defining a function does not run its body — the body only runs when the function is called.

In [ ]:
def greet(name):
    return f"Hello, {name}!"

print(greet("Alice"))   # Hello, Alice!
print(greet("Bob"))     # Hello, Bob!

# A function with no return statement returns None
def say_hello():
    print("Hello!")

result = say_hello()   # prints "Hello!"
print(result)          # None

In [ ]:
# Functions replace duplicated logic
def celsius_to_fahrenheit(c):
    return c * 9 / 5 + 32

for temp in [0, 20, 37, 100]:
    print(f"{temp}°C = {celsius_to_fahrenheit(temp):.1f}°F")

## Parameters and Arguments

**Parameters** are the names listed in the function definition. **Arguments** are the actual values you pass when calling the function. Python is flexible about how you pass them.

### Positional arguments
Matched to parameters left to right, in order. Most common.

### Keyword arguments
Passed by name — order does not matter. Make call sites more readable, especially when a function has many parameters.

You can mix both in one call: positional arguments must come before keyword arguments.

In [ ]:
def describe_pet(name, animal, age):
    print(f"{name} is a {age}-year-old {animal}.")

# Positional — order matters
describe_pet("Rex", "dog", 3)

# Keyword — order is irrelevant
describe_pet(age=5, name="Whiskers", animal="cat")

# Mixed — positional first, then keyword
describe_pet("Goldie", age=2, animal="fish")

## Default Parameter Values

You can give a parameter a default value. If the caller does not pass that argument, the default is used. Parameters with defaults must come **after** any parameters without defaults.

**Important gotcha**: never use a mutable object (list, dict, set) as a default value. Default values are evaluated **once** when the function is defined, not on each call — so a mutable default is shared across all calls. Use `None` as the default and create the mutable object inside the function body.

In [ ]:
def send_email(to, subject, cc=None, priority="normal"):
    print(f"To: {to}")
    print(f"Subject: {subject}")
    print(f"CC: {cc or 'none'}")
    print(f"Priority: {priority}")
    print()

send_email("alice@example.com", "Meeting tomorrow")
send_email("bob@example.com", "Urgent issue", priority="high")
send_email("carol@example.com", "FYI", cc="manager@example.com")

In [ ]:
# The mutable default gotcha
def bad_append(value, items=[]):   # DON'T DO THIS
    items.append(value)
    return items

print(bad_append(1))   # [1]   — looks fine
print(bad_append(2))   # [1, 2] — the list is shared between calls!
print(bad_append(3))   # [1, 2, 3]

# The correct pattern: default to None, create inside
def good_append(value, items=None):
    if items is None:
        items = []
    items.append(value)
    return items

print(good_append(1))  # [1]
print(good_append(2))  # [2] — fresh list each time

## Return Values

`return` immediately exits the function and hands back a value to the caller. A function can have multiple `return` statements — the first one reached ends execution. You can return any Python object: a number, string, list, another function, or `None`.

To return multiple values, separate them with commas. Python packs them into a **tuple**, which you can unpack at the call site.

In [ ]:
# Early return — exit as soon as the answer is known
def first_negative(numbers):
    for n in numbers:
        if n < 0:
            return n   # exit immediately
    return None        # no negative found

print(first_negative([4, 7, -2, 9]))  # -2
print(first_negative([1, 2, 3]))      # None

In [ ]:
# Returning multiple values — Python packs them into a tuple
def min_max(numbers):
    return min(numbers), max(numbers)

low, high = min_max([4, 1, 9, 2, 7])
print(f"Min: {low}, Max: {high}")   # Min: 1, Max: 9

# Or capture the tuple without unpacking
result = min_max([4, 1, 9, 2, 7])
print(result)          # (1, 9)
print(type(result))    # <class 'tuple'>

## `*args` and `**kwargs`

Sometimes you want a function to accept a variable number of arguments without listing them all in advance.

- **`*args`** — collects any extra **positional** arguments into a tuple.
- **`**kwargs`** — collects any extra **keyword** arguments into a dictionary.

The names `args` and `kwargs` are just conventions — what matters is the `*` and `**` prefix. You can name them anything (`*numbers`, `**options`), but `*args`/`**kwargs` are universally recognised.

The full parameter order rule: `positional`, `*args`, `keyword-only`, `**kwargs`.

In [ ]:
# *args — variable number of positional arguments
def total(*numbers):
    return sum(numbers)

print(total(1, 2))          # 3
print(total(1, 2, 3, 4, 5)) # 15
print(total())              # 0 — sum of nothing

# args is a regular tuple inside the function
def show_args(*args):
    print(type(args), args)

show_args("a", "b", "c")

In [ ]:
# **kwargs — variable number of keyword arguments
def build_profile(**kwargs):
    for key, value in kwargs.items():
        print(f"  {key}: {value}")

build_profile(name="Alice", role="engineer", team="platform", years=4)

# kwargs is a regular dict inside the function
def show_kwargs(**kwargs):
    print(type(kwargs), kwargs)

show_kwargs(x=1, y=2)

In [ ]:
# Combining all parameter kinds
def log(level, *messages, sep=" | ", **context):
    joined = sep.join(str(m) for m in messages)
    ctx = " ".join(f"{k}={v}" for k, v in context.items())
    print(f"[{level.upper()}] {joined}  {ctx}")

log("info", "Server started", "Listening on port 8080", host="localhost", pid=1234)
log("error", "Connection failed", sep=" — ", retry=True)

In [ ]:
# Unpacking with * and ** when calling
def add(a, b, c):
    return a + b + c

numbers = [1, 2, 3]
print(add(*numbers))         # unpack list as positional args → 6

values = {"a": 10, "b": 20, "c": 30}
print(add(**values))         # unpack dict as keyword args → 60

## Scope — Local, Global, and `nonlocal`

**Scope** determines which parts of your code can see and modify a variable.

- **Local scope**: variables created inside a function. They exist only for the duration of that call and are invisible outside.
- **Global scope**: variables defined at the top level of a module. Visible everywhere in that module.

Python looks up a name using the **LEGB rule**: Local → Enclosing → Global → Built-in. It searches each layer in that order and uses the first match.

A function can **read** a global variable without any special declaration. But to **reassign** it (make the name point to something new), you must declare `global name` inside the function. Avoid this in practice — functions that modify global state are harder to test and reason about. Prefer passing values in as arguments and returning them out.

In [ ]:
x = 100   # global

def show_x():
    print(x)   # reads global — fine

def reset_x():
    global x   # declare intent to rebind the global name
    x = 0

show_x()    # 100
reset_x()
show_x()    # 0

def local_x():
    x = 999   # creates a NEW local variable — does not touch the global
    print("local:", x)

local_x()   # local: 999
show_x()    # 0 — global unchanged

In [ ]:
# nonlocal — modifies a variable in the enclosing (but not global) scope
def make_counter():
    count = 0

    def increment():
        nonlocal count   # refers to make_counter's local 'count'
        count += 1
        return count

    return increment

counter = make_counter()
print(counter())  # 1
print(counter())  # 2
print(counter())  # 3

# A second counter is independent — its own enclosing scope
other = make_counter()
print(other())    # 1

## Docstrings

A **docstring** is a string literal placed as the very first statement of a function body. Python stores it in the function's `__doc__` attribute and tools like `help()`, IDEs, and documentation generators read it automatically.

Use triple quotes for multi-line docstrings. The first line should be a short, imperative summary. If more detail is needed, leave a blank line and add parameters, return values, and examples.

In [ ]:
def clamp(value, low, high):
    """Return value constrained to the range [low, high].

    If value is below low, return low.
    If value is above high, return high.
    Otherwise return value unchanged.

    Args:
        value: The number to clamp.
        low:   The minimum allowed value.
        high:  The maximum allowed value.

    Returns:
        The clamped value.

    Examples:
        >>> clamp(5, 0, 10)   # 5
        >>> clamp(-3, 0, 10)  # 0
        >>> clamp(15, 0, 10)  # 10
    """
    return max(low, min(value, high))

print(clamp(5, 0, 10))    # 5
print(clamp(-3, 0, 10))   # 0
print(clamp(15, 0, 10))   # 10

help(clamp)   # reads the docstring

## Lambda Functions

A **lambda** is a small anonymous function defined in a single expression. The syntax is:

```python
lambda parameters: expression
```

The expression is evaluated and implicitly returned — there is no `return` keyword. Lambdas are limited to a single expression, which means no assignments, no statements, no multi-line logic.

Their main use is as a short, throwaway function passed as an argument — most often to `sorted()`, `map()`, `filter()`, or similar functions that accept a callable. For anything more than a few characters of logic, prefer a named function: it is easier to read, test, and reuse.

In [ ]:
# Lambda as a sort key
people = [
    {"name": "Alice", "age": 30},
    {"name": "Bob",   "age": 25},
    {"name": "Carol", "age": 35},
]

# Sort by age
by_age = sorted(people, key=lambda p: p["age"])
for p in by_age:
    print(p["name"], p["age"])

print()

# Sort strings by their length, then alphabetically
words = ["banana", "fig", "apple", "kiwi", "date"]
print(sorted(words, key=lambda w: (len(w), w)))

In [ ]:
# map() — apply a function to every item, returns an iterator
prices = [9.99, 14.50, 4.25, 22.00]
discounted = list(map(lambda p: round(p * 0.9, 2), prices))
print(discounted)

# filter() — keep items where the function returns True
numbers = range(-5, 6)
positives = list(filter(lambda n: n > 0, numbers))
print(positives)

## Functions as First-Class Objects

In Python, functions are **first-class objects** — they are values, just like integers or strings. This means you can:

- Assign a function to a variable
- Pass a function as an argument to another function
- Return a function from a function
- Store functions in lists or dictionaries

A function that takes another function as an argument, or returns one, is called a **higher-order function**.

In [ ]:
# Assign a function to a variable — the name is just a label
def square(n):
    return n ** 2

transform = square          # no parentheses — we're pointing at the function, not calling it
print(transform(5))         # 25

# Store functions in a data structure
import math

operations = {
    "square": lambda n: n ** 2,
    "sqrt":   math.sqrt,
    "double": lambda n: n * 2,
}

for name, fn in operations.items():
    print(f"{name}(9) = {fn(9)}")

In [ ]:
# Higher-order function — takes a function as an argument
def apply_twice(fn, value):
    return fn(fn(value))

print(apply_twice(lambda n: n * 2, 3))    # 12 — doubled twice
print(apply_twice(str.upper, "hello"))    # won't work: str.upper needs a string receiver
print(apply_twice(lambda s: s + "!", "hello"))  # hello!!

In [ ]:
# Returning a function — closure captures the enclosing scope
def make_multiplier(factor):
    def multiply(n):
        return n * factor   # 'factor' is captured from the enclosing call
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(7))   # 14
print(triple(7))   # 21

# Both functions remember their own 'factor'
print([double(n) for n in range(1, 6)])   # [2, 4, 6, 8, 10]
print([triple(n) for n in range(1, 6)])   # [3, 6, 9, 12, 15]

## Recursion

A function is **recursive** when it calls itself. Each call works on a smaller piece of the problem until it reaches a **base case** — a condition where it returns a result directly without recursing further. Without a base case, the function calls itself forever and Python raises a `RecursionError`.

Recursion is a natural fit for problems that are self-similar — trees, nested structures, divide-and-conquer algorithms. For straightforward iteration though, a loop is usually clearer and faster. Python's default recursion limit is 1000 levels deep.

In [ ]:
# Classic example: factorial
# factorial(n) = n * factorial(n-1), base case: factorial(0) = 1
def factorial(n):
    if n == 0:            # base case
        return 1
    return n * factorial(n - 1)   # recursive case

for i in range(7):
    print(f"{i}! = {factorial(i)}")

In [ ]:
# Recursion shines on nested structures
def flatten(nested):
    """Flatten an arbitrarily deep nested list into a flat list."""
    result = []
    for item in nested:
        if isinstance(item, list):
            result.extend(flatten(item))   # recurse into sub-lists
        else:
            result.append(item)
    return result

data = [1, [2, 3], [4, [5, 6]], [[[7]]]]
print(flatten(data))   # [1, 2, 3, 4, 5, 6, 7]

## Summary

- Define functions with `def name(params):`. A function that reaches the end without `return` implicitly returns `None`.
- **Positional arguments** are matched by order; **keyword arguments** are matched by name. Keyword arguments make call sites self-documenting.
- **Default parameter values** make arguments optional. Never use a mutable object (list, dict) as a default — use `None` and create the object inside the function.
- `return` exits the function and returns a value. Return multiple values with a comma — Python packs them into a tuple.
- **`*args`** collects extra positional arguments into a tuple. **`**kwargs`** collects extra keyword arguments into a dict. Use `*seq` and `**mapping` to unpack them at the call site.
- **Scope follows LEGB**: Local → Enclosing → Global → Built-in. Use `global` to rebind a global variable inside a function, and `nonlocal` to rebind an enclosing scope variable. Prefer passing and returning values over modifying outer scope.
- **Docstrings** document what a function does. Put them as the first statement, use triple quotes, and keep the first line a concise imperative summary.
- **Lambdas** are anonymous single-expression functions — useful as throwaway arguments to `sorted()`, `map()`, and `filter()`.
- Functions are **first-class objects** — they can be assigned, passed, returned, and stored. Higher-order functions and closures are built on this.
- **Recursion** is a function calling itself with a smaller input until a base case is reached. Natural for nested/self-similar structures; loops are usually preferable for simple iteration.